In [1]:
import sys
from pathlib import Path
import numpy as np
import torch
import pandas as pd
from chronos import Chronos2Pipeline

REPO_ROOT = Path.cwd().parent.parent
BACKEND_DIR = REPO_ROOT / "backend"
sys.path.insert(0, str(BACKEND_DIR))
sys.path.insert(0, str(Path.cwd()))

from _pool_common import (
    load_pool_data,
    backtest_21d_rolling,
    compute_metrics_averaged_over_windows,
    metrics_to_parquet,
    TEST_SIZE,
    FORECAST_HORIZON,
    ROLLING_STEP,
    MIN_CONTEXT_CHRONOS,
    ARTIFACTS_DIR,
    TICKERS,
)

MODEL_ID = "amazon/chronos-2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

c:\capstone_project_unfc\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print(f"Loading {MODEL_ID} on {DEVICE}...")
pipeline = Chronos2Pipeline.from_pretrained(
    MODEL_ID,
    device_map=DEVICE,
    torch_dtype=torch.float32,
)

def get_forecast_chronos(context: pd.Series, horizon: int):
    """Predict returns with Chronos, then compound to price path; return list of `horizon` price forecasts."""
    try:
        context = context.astype(float)
        idx = pd.date_range(context.index.min(), context.index.max(), freq="B")
        prices_regular = context.reindex(idx).ffill().bfill().dropna()
        if len(prices_regular) < 3:
            return []
        # Convert price to simple returns: r_t = (P_t - P_{t-1}) / P_{t-1}
        prices_arr = np.asarray(prices_regular.values.flatten(), dtype=float)
        returns_arr = (prices_arr[1:] - prices_arr[:-1]) / (prices_arr[:-1] + 1e-12)
        timestamps = prices_regular.index[1:]
        input_df = pd.DataFrame({
            "item_id": ["x"] * len(returns_arr),
            "timestamp": timestamps,
            "target": returns_arr,
        })
        forecast_df = pipeline.predict_df(
            input_df,
            prediction_length=horizon,
            quantile_levels=[0.5],
            id_column="item_id",
            timestamp_column="timestamp",
            target="target",
            validate_inputs=True,
        )
        qcol = "0.5" if "0.5" in forecast_df.columns else "predictions"
        if qcol not in forecast_df.columns:
            qcol = forecast_df.columns[-1]
        return_forecasts = np.asarray(forecast_df[qcol]).ravel()[:horizon]
        # Compound returns to price: P_0 = last price, P_{k} = P_{k-1} * (1 + r_k)
        last_price = float(prices_arr[-1])
        price_forecasts = []
        p = last_price
        for r in return_forecasts:
            p = p * (1.0 + float(r))
            price_forecasts.append(p)
        return price_forecasts
    except Exception as e:
        print(f"Chronos forecast error: {e}")
        return []

Loading amazon/chronos-2 on cpu...


`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!


In [3]:
stacked = load_pool_data()
print(stacked.groupby("symbol").size())
stacked.head(10)

symbol
AAPL     1256
AMZN     1256
GOOGL    1256
JNJ      1256
JPM      1256
MSFT     1256
NVDA     1256
SPY      1256
WMT      1256
XOM      1256
dtype: int64


,timestamp,symbol,close
0,2021-03-01,AAPL,127.790001
1,2021-03-02,AAPL,125.120003
2,2021-03-03,AAPL,122.059998
3,2021-03-04,AAPL,120.129997
4,2021-03-05,AAPL,121.419998
5,2021-03-08,AAPL,116.360001
6,2021-03-09,AAPL,121.089996
7,2021-03-10,AAPL,119.980003
8,2021-03-11,AAPL,121.959999
9,2021-03-12,AAPL,121.029999


In [4]:
# Diagnostic: run one Chronos forecast and inspect output (delete or skip after debugging)
_sym = TICKERS[0]
_grp = stacked[stacked["symbol"] == _sym]
_prices = _grp.set_index("timestamp")["close"].astype(float).dropna().sort_index()
_context = _prices.iloc[-200:]  # last 200 points
_input_df = pd.DataFrame({
    "item_id": ["x"] * len(_context),
    "timestamp": _context.index,
    "target": _context.values,
})
print("Input shape:", _input_df.shape, "| timestamp dtype:", _input_df["timestamp"].dtype)
try:
    _out = pipeline.predict_df(_input_df, prediction_length=21, quantile_levels=[0.5],
                               id_column="item_id", timestamp_column="timestamp", target="target")
    print("Forecast shape:", _out.shape, "| columns:", list(_out.columns))
    print(_out.head(3))
    _got = get_forecast_chronos(_context, 21)
    print("get_forecast_chronos returned len:", len(_got), "| first 3:", _got[:3] if _got else "[]")
except Exception as e:
    print("Error:", type(e).__name__, e)

Input shape: (200, 3) | timestamp dtype: datetime64[ms]
Error: ValueError Could not infer frequency for series x


In [5]:
# 7-day-ahead direct forecast; 60-day test window, rolling by 7 days (held-out test assets only)
model_name = "chronos"
all_preds = []
for sym in TICKERS:
    grp = stacked[stacked["symbol"] == sym]
    if grp.empty:
        continue
    prices = grp.set_index("timestamp")["close"].astype(float).dropna().sort_index()
    if len(prices) < TEST_SIZE + MIN_CONTEXT_CHRONOS:
        continue
    pred = backtest_21d_rolling(
        prices, TEST_SIZE, FORECAST_HORIZON, ROLLING_STEP,
        MIN_CONTEXT_CHRONOS,
        get_forecast_fn=get_forecast_chronos,
    )
    if pred.empty:
        continue
    pred["symbol"] = sym
    all_preds.append(pred)

pred_chronos = pd.concat(all_preds, ignore_index=True) if all_preds else pd.DataFrame(columns=["timestamp", "y_true", "y_pred", "window_ix", "symbol"])
print(pred_chronos.groupby("symbol").size() if not pred_chronos.empty else "No predictions (all test symbols skipped or backtest returned empty).")
pred_chronos.head()

c:\capstone_project_unfc\env\Lib\site-packages\chronos\chronos2\dataset.py:89: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  task_target = torch.from_numpy(task_target)


symbol
AAPL     56
AMZN     56
GOOGL    56
JNJ      56
JPM      56
MSFT     56
NVDA     56
SPY      56
WMT      56
XOM      56
dtype: int64


,timestamp,y_true,y_pred,window_ix,symbol
0,2025-12-02,286.190002,283.515019,0,AAPL
1,2025-12-03,284.149994,283.878417,0,AAPL
2,2025-12-04,280.700012,284.243799,0,AAPL
3,2025-12-05,278.779999,284.622328,0,AAPL
4,2025-12-08,277.890015,284.972207,0,AAPL


In [6]:
metrics_rows = []
for sym in pred_chronos["symbol"].unique():
    sub = pred_chronos[pred_chronos["symbol"] == sym]
    m = compute_metrics_averaged_over_windows(sub)
    metrics_rows.append({"model": model_name, "symbol": sym, **m})
m_overall = compute_metrics_averaged_over_windows(pred_chronos)
metrics_rows.append({"model": model_name, "symbol": "overall", **m_overall})

metrics_df = pd.DataFrame(metrics_rows)
print(metrics_df.to_string())
metrics_to_parquet(metrics_rows, ARTIFACTS_DIR / "metrics_chronos_pool.parquet")
print("Saved:", ARTIFACTS_DIR / "metrics_chronos_pool.parquet")

      model   symbol        MAE       RMSE    MAPE_%
0   chronos     AAPL   7.082376   7.961544  2.686139
1   chronos     MSFT  11.264566  12.269155  2.547827
2   chronos    GOOGL   8.209629   9.284309  2.592246
3   chronos     AMZN   8.949377   9.923145  3.991182
4   chronos      JPM   7.003046   7.952593  2.243015
5   chronos      JNJ   3.943893   4.383735  1.771537
6   chronos      WMT   2.472310   2.797631  2.057891
7   chronos      SPY   6.822969   7.561823  0.996615
8   chronos      XOM   4.515252   4.858258  3.299225
9   chronos     NVDA   4.290693   5.059722  2.363521
10  chronos  overall   6.455411   8.505310  2.454920


Saved: C:\capstone_project_unfc\model\experiments-pool\artifacts\metrics_chronos_pool.parquet
